In [ ]:
async def grade_assignments(assigner: LlmApiClusterAssigner):
    sizes = {'elicit': 12, 'cursor': 34, 'bridge': 20, 'pico': 34}
    results = await assigner.assign(DATASET["items"], DATASET["clusters"])
    index = 0
    for category, size in sizes.items():
        correct = sum(1 for i, r in enumerate(results[index:index+size]) if r is not None and r[0] == DATASET["labels"][index + i] and r[0])
        incorrect = sum(1 for i, r in enumerate(results[index:index+size]) if r is not None and r[0] == DATASET["labels"][index + i] and not r[0])
        string = (f"{category}: 0.{int(100 * (correct + incorrect) / size)}")
        if sum(r[0] for r in results[index:index+size] if r is not None) > 0:
            string += f", correct prec 0.{int(100 * correct / sum(r[0] for r in results[index:index+size] if r is not None))}"
        if sum(not r[0] for r in results[index:index+size] if r is not None) > 0:
            string += f", incorrect prec 0.{int(100 * incorrect / sum(not r[0] for r in results[index:index+size] if r is not None))}"
        index += size
        print(string)
    print(f"overall: 0.{sum(1 for i, r in enumerate(results) if r is not None and r[0] == DATASET['labels'][i])}")

(await grade_assignments(LlmApiClusterAssigner.from_o3_mini()))
(await grade_assignments(LlmApiClusterAssigner.from_sonnet_37_thinking()))
(await grade_assignments(LlmApiClusterAssigner(
    system_prompt=None,
    max_new_tokens=8192,
    temperature=1,
    model_options=[
            ModelOption(
                provider="openai",
                model_name="o4-mini",
                reasoning_effort="medium",
            ),
        ]
)))

In [ ]:

def new_assign_prompt_fn(item: str, cluster: str):
    return """
You are given a cluster label C and an item I.
Your task is to perform membership testing: determine whether the description of C matches I.

The cluster label may come with examples, which you shouldn't treat as requirements; they simply give a feel for items that would belong.

Return two lines in the following exact format:
- ANSWER: 
- EXPLANATION: 

Here is your input:
C: {cluster}
I: {item}
""".strip().format(cluster=cluster, item=item)

def new_assign_prompt_fn_2(item: str, cluster: str):
    return """
You are given a label C and an item I.
Your task is to perform membership testing: determine whether the description of C matches I.

Return two lines in the following exact format:
- ANSWER: 
- EXPLANATION: 

Here is your input:
C: {cluster}
I: {item}
""".strip().format(cluster=item, item=cluster)

def new_assign_prompt_fn_3(item: str, cluster: str):
    return """
You are given a detailed item A and a broader item B.
Your task is to perform membership testing: determine whether the description of A matches B.

Return two lines in the following exact format:
- ANSWER: 
- EXPLANATION: 

Here is your input:
A: {item}
B: {cluster}
""".strip().format(cluster=cluster, item=item)

In [ ]:
(await grade_assignments(LlmApiClusterAssigner(
    system_prompt=None,
    max_new_tokens=8192,
    temperature=1,
    model_options=[
            ModelOption(
                provider="openai",
                model_name="o4-mini",
                reasoning_effort="medium",
            ),
        ],
    assign_prompt_fn=new_assign_prompt_fn
)))


(await grade_assignments(LlmApiClusterAssigner(
    system_prompt=None,
    max_new_tokens=8192,
    temperature=1,
    model_options=[
            ModelOption(
                provider="openai",
                model_name="o4-mini",
                reasoning_effort="medium",
            ),
        ],
    assign_prompt_fn=new_assign_prompt_fn_2
)))


(await grade_assignments(LlmApiClusterAssigner(
    system_prompt=None,
    max_new_tokens=8192,
    temperature=1,
    model_options=[
            ModelOption(
                provider="openai",
                model_name="o4-mini",
                reasoning_effort="medium",
            ),
        ],
    assign_prompt_fn=new_assign_prompt_fn_3
)))